# 046 — Sistemas de recomendación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Problema:** matriz usuario×ítem casi vacía (>99 %); el objetivo es el ranking top-k por
usuario, no rellenar celdas. Feedback explícito (ratings) vs. implícito (clics: la
ausencia no es rechazo, es no exposición).

**Tres enfoques:**

- **Colaborativo (vecindad):** similitud coseno entre usuarios o ítems;
  `r̂(u,i) = Σ sim·r / Σ|sim|`. Sin features, pero ciego ante ítems/usuarios nuevos.
- **Contenido:** atributos del ítem + perfil del usuario; resuelve cold-start de ítem,
  explicable, tiende a la burbuja.
- **Factorización:** `r̂ = μ + b_u + b_i + p_u·q_i`, optimizada con SGD/ALS solo sobre
  celdas observadas + regularización λ. Factores latentes = embeddings primitivos.

**Evaluación:** split temporal por usuario; Precision@k, Recall@k, NDCG@k (descuento por
posición), cobertura y diversidad. Baseline obligatorio: popularidad. Todo sistema serio
es híbrido.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** ‖Ana‖ = √42 ≈ 6.481. sim(Ana,Beto) = 47/(6.481·√54) ≈ 0.987;
sim(Ana,Carla) = 18/(6.481·√30) ≈ 0.507; sim(Ana,Dani) = 37/(6.481·√33) ≈ 0.994.
r̂(Ana,D) = (0.987·4 + 0.507·2 + 0.994·5)/2.488 ≈ **3.99**. Si Carla puntuara D = 5:
numerador = 3.948 + 2.535 + 4.970 = 11.45 → r̂ ≈ 4.60: sube, pero poco — su similitud
baja (0.507) limita su influencia. El coseno sin centrar además la cuenta como
"parecida a medias"; centrando por usuario, sus gustos opuestos pesarían en contra.

**Ejercicio 2.** Aciertos en top-5: {i₂, i₅} → Precision@5 = 2/5 = **0.40**;
Recall@5 = 2/4 = **0.50**. En top-3 solo i₂ → Precision@3 = 1/3 ≈ **0.33**.
Recall@k es no decreciente en k (solo puede sumar aciertos); comparar sistemas exige
fijar el mismo k — y k debe ser el tamaño de lista que el producto realmente muestra.

**Ejercicio 3.** DCG(R₁) = 1/log₂2 + 0 + 0 + 1/log₂5 = 1 + 0.431 = **1.431**.
DCG(R₂) = 1/log₂3 + 1/log₂4 = 0.631 + 0.5 = **1.131**. Gana R₁: mismos aciertos, pero
uno en la posición 1. El descuento logarítmico codifica que la atención del usuario se
concentra arriba: un acierto en la posición 1 vale el doble que en la 4.

**Ejercicio 4.** La limitación "la representación es bag-of-words" implica que solo se
recomienda lo que comparte términos exactos con el perfil: sin sinónimos ni semántica, el
usuario queda encerrado en su vocabulario — la misma sobre-especialización del filtrado
por contenido, que los métodos colaborativos (que usan señal de OTROS usuarios) y los
embeddings densos (parte 08) atacan por caminos distintos.


In [ ]:
result = run_lab("retrieval", seed=46)
assert result["kind"] == "retrieval"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — coseno y predicción por vecindad
import math

usuarios = {"Beto": [5, 5, 2], "Carla": [1, 2, 5], "Dani": [4, 4, 1]}
ana = [5, 4, 1]
rating_d = {"Beto": 4, "Carla": 2, "Dani": 5}

def coseno(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

sims = {u: coseno(ana, v) for u, v in usuarios.items()}
pred = sum(sims[u] * rating_d[u] for u in sims) / sum(sims.values())
for u, s in sims.items():
    print(f"sim(Ana, {u}) = {s:.3f}")
print(f"r̂(Ana, D) = {pred:.2f}")


In [ ]:
# Ejercicios 2 y 3 — métricas de ranking
import math

top5 = ["i1", "i2", "i3", "i4", "i5"]
relevantes = {"i2", "i5", "i7", "i9"}
hits5 = [i for i in top5 if i in relevantes]
print(f"Precision@5 = {len(hits5) / 5:.2f}  Recall@5 = {len(hits5) / len(relevantes):.2f}")
print(f"Precision@3 = {len([i for i in top5[:3] if i in relevantes]) / 3:.2f}")

def dcg(rels):
    return sum(r / math.log2(i + 1) for i, r in enumerate(rels, start=1))

print(f"DCG R1 = {dcg([1, 0, 0, 1]):.3f}   DCG R2 = {dcg([0, 1, 1, 0]):.3f}")
# Mismos aciertos, gana quien los pone arriba: la posición es parte del valor.


## Reflexión

1. El laboratorio rankea documentos para una consulta con bag-of-words y score de
   solapamiento. Mapea cada pieza (consulta, documentos, score, ranking) a su equivalente
   en un recomendador basado en contenido. ¿Qué juega el papel del "perfil del usuario"?
2. En el resultado del laboratorio, dos documentos empatan con score 0.0. En un
   recomendador real, ¿cómo desempatarías el final del ranking y por qué "orden de
   aparición" es una mala respuesta?
3. ¿Por qué un split aleatorio de interacciones infla las métricas de un recomendador y
   qué evento del mundo real simula correctamente el split temporal?
